# Price exploration: reference trading price (30-minute) and market clearing price (5-minute)

Years compared: 2024, 2025 and 2026 (2026 runs to the latest extract only).

Conventions in the AEMO files:
- Times are Perth time (AWST, UTC+8, no daylight saving) and label the **start** of each interval.
- Dates are day-first (`01/02/2024` is 1 February).
- A trading day runs 08:00 to 08:00, so a "day" here is not a calendar day.
- Files are sorted newest first, so everything is sorted by time after loading.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW = Path("../data/raw")        # notebooks/ sits one level below the project root
YEARS = [2024, 2025, 2026]

C30, C5 = "#1f77b4", "#d95f02"   # blue = 30-minute, orange = 5-minute
plt.rcParams.update({
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Functions

In [ ]:
# ---------- loading ----------

def _read_prices(path: Path, time_col: str, price_col: str) -> pd.Series:
    """Read one AEMO price file into a Series of prices indexed by interval start."""
    df = pd.read_csv(path, usecols=[time_col, price_col])
    interval = pd.to_datetime(df[time_col], format="%d/%m/%Y %H:%M:%S")
    index = pd.DatetimeIndex(interval, name="interval")
    return pd.Series(df[price_col].to_numpy(), index=index, name="price").sort_index()


def load_reference_price(year: int) -> pd.Series:
    """Half-hourly reference trading price ($/MWh)."""
    path = RAW / "reference-trading-price-csv" / f"ReferenceTradingPrice-{year}.csv"
    return _read_prices(path, "Trading Interval", "Reference Trading Price")


def load_market_clearing(year: int) -> pd.Series:
    """Five-minute energy clearing price ($/MWh)."""
    path = RAW / "market-clearing-prices-csv" / f"MarketClearingPrices-{year}.csv"
    return _read_prices(path, "Dispatch Interval", "Energy Clearing Price")


# ---------- trading days and checks ----------

def trading_day(index: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """Trading date of each interval. A trading day runs 08:00 to 08:00."""
    return (index - pd.Timedelta(hours=8)).normalize()


def check_series(s: pd.Series, per_day: int) -> dict:
    """Basic sanity checks. per_day is 48 for half-hourly data and 288 for 5-minute."""
    days = trading_day(s.index).nunique()
    return {
        "first": s.index.min(),
        "last": s.index.max(),
        "rows": len(s),
        "expected_rows": days * per_day,
        "duplicates": int(s.index.duplicated().sum()),
        "missing_prices": int(s.isna().sum()),
    }


# ---------- analysis ----------

def avg_by_time_of_day(s: pd.Series) -> pd.Series:
    """Mean price at each time of day, indexed by hour (0 to 23.98)."""
    hour = s.index.hour + s.index.minute / 60
    return s.groupby(hour).mean()


def daily_spread(s: pd.Series) -> pd.Series:
    """Max price minus min price for each trading day."""
    g = s.groupby(trading_day(s.index))
    return g.max() - g.min()


def top_share(spread: pd.Series, frac: float = 0.1) -> float:
    """Share of the total spread that comes from the biggest `frac` of days."""
    n = int(round(len(spread) * frac))
    return spread.nlargest(n).sum() / spread.sum()


def get_day(s: pd.Series, day) -> pd.Series:
    """Prices for one trading day (08:00 on `day` to 08:00 the next day)."""
    start = pd.Timestamp(day) + pd.Timedelta(hours=8)
    return s[(s.index >= start) & (s.index < start + pd.Timedelta(days=1))]


# ---------- plots ----------

def plot_year(year: int, p30: pd.Series, p5: pd.Series) -> None:
    """Three panels for one year: average daily shape, daily spread, biggest day."""
    sp30, sp5 = daily_spread(p30), daily_spread(p5)
    worst = sp5.idxmax()

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 4))
    fig.suptitle(f"{year}: {p5.index.min():%d %b %Y} to {p5.index.max():%d %b %Y}", y=1.03)

    # 1. average price by time of day
    a30, a5 = avg_by_time_of_day(p30), avg_by_time_of_day(p5)
    ax1.plot(a5.index, a5.values, color=C5, label="5-minute")
    ax1.plot(a30.index, a30.values, color=C30, label="30-minute")
    ticks = list(range(0, 24, 3))
    ax1.set_xticks(ticks)
    ax1.set_xticklabels([f"{h:02d}:00" for h in ticks])
    ax1.set_title("Average price by time of day")
    ax1.set_xlabel("Time of day (AWST)")
    ax1.set_ylabel("$/MWh")
    ax1.legend()

    # 2. daily spread through the year
    ax2.plot(sp5.index, sp5.values, color=C5, linewidth=0.8, label="5-minute")
    ax2.plot(sp30.index, sp30.values, color=C30, linewidth=0.8, label="30-minute")
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax2.set_title("Daily spread (max - min)")
    ax2.set_ylabel("$/MWh")
    ax2.legend()

    # 3. the day with the biggest 5-minute spread
    d5, d30 = get_day(p5, worst), get_day(p30, worst)
    ax3.plot(d5.index, d5.values, color=C5, linewidth=1, label="5-minute")
    ax3.step(d30.index, d30.values, where="post", color=C30, linewidth=1, label="30-minute")
    ax3.xaxis.set_major_locator(mdates.HourLocator(interval=4))
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax3.set_title(f"Biggest day: {worst:%d %b %Y} (08:00 to 08:00)")
    ax3.set_ylabel("$/MWh")
    ax3.legend()

    fig.tight_layout()


def plot_profile_comparison(profiles30: dict, profiles5: dict) -> None:
    """Average price by time of day, one line per year, for each interval length."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for ax, profiles, title in [(axes[0], profiles30, "30-minute"), (axes[1], profiles5, "5-minute")]:
        colours = plt.cm.Blues(np.linspace(0.4, 0.95, len(profiles)))
        for (year, prof), colour in zip(profiles.items(), colours):
            ax.plot(prof.index, prof.values, color=colour, label=str(year))
        ticks = list(range(0, 24, 3))
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{h:02d}:00" for h in ticks])
        ax.set_title(f"Average price by time of day ({title})")
        ax.set_xlabel("Time of day (AWST)")
        ax.legend()
    axes[0].set_ylabel("$/MWh")
    fig.tight_layout()

## Load and check the data

In [ ]:
data = {}
for year in YEARS:
    data[year] = {
        "p30": load_reference_price(year),
        "p5": load_market_clearing(year),
    }

checks = pd.DataFrame({
    (year, label): check_series(data[year][key], per_day)
    for year in YEARS
    for label, key, per_day in [("30-minute", "p30", 48), ("5-minute", "p5", 288)]
}).T
checks

`rows` should equal `expected_rows` with no duplicates or missing prices. The 2026 file stops at the latest extract, so its last day is incomplete.

## Summary by year

In [ ]:
rows = []
for year in YEARS:
    for label, key in [("30-minute", "p30"), ("5-minute", "p5")]:
        s = data[year][key]
        sp = daily_spread(s)
        rows.append({
            "year": year,
            "interval": label,
            "days": len(sp),
            "price mean": s.mean(),
            "price min": s.min(),
            "price max": s.max(),
            "spread mean": sp.mean(),
            "spread median": sp.median(),
            "spread max": sp.max(),
            "top 10% of days, % of total spread": 100 * top_share(sp),
        })

summary = pd.DataFrame(rows).set_index(["year", "interval"])
summary.round(1)

## Like-for-like spread

2026 only covers part of the year, so this compares each year over the same trading days (1 January to the last complete day in 2026).

In [ ]:
spreads = {
    (year, label): daily_spread(data[year][key])
    for year in YEARS
    for label, key in [("30-minute", "p30"), ("5-minute", "p5")]
}

last_day = spreads[(YEARS[-1], "5-minute")].index.max() - pd.Timedelta(days=1)  # drop the partial last day

rows = []
for (year, label), sp in spreads.items():
    same_period = sp[sp.index <= last_day.replace(year=year)]
    rows.append({
        "year": year,
        "interval": label,
        "days": len(same_period),
        "spread mean": same_period.mean(),
        "spread median": same_period.median(),
    })

pd.DataFrame(rows).set_index(["year", "interval"]).round(1)

## Plots by year

In [ ]:
for year in YEARS:
    plot_year(year, data[year]["p30"], data[year]["p5"])
    plt.show()

## Average daily shape, years compared

In [ ]:
profiles30 = {year: avg_by_time_of_day(data[year]["p30"]) for year in YEARS}
profiles5 = {year: avg_by_time_of_day(data[year]["p5"]) for year in YEARS}
plot_profile_comparison(profiles30, profiles5)
plt.show()